# Compare Prediction with validation data

Plotted on this [map](https://terriamap.p.niva.no/#start=%7B%22version%22%3A%228.0.0%22%2C%22initSources%22%3A%5B%7B%22stratum%22%3A%22user%22%2C%22models%22%3A%7B%22%2F%2FMarint+Natur+Kart%22%3A%7B%22isOpen%22%3Atrue%2C%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FMarint+Natur+Kart%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FBunntyper+Validering%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%2FBunntyper+Validering%22%5D%2C%22type%22%3A%22wfs%22%7D%2C%22%2F%22%3A%7B%22type%22%3A%22group%22%7D%2C%22%2F%2FBunntyper+Validering%22%3A%7B%22knownContainerUniqueIds%22%3A%5B%22%2F%22%5D%2C%22type%22%3A%22group%22%7D%7D%2C%22workbench%22%3A%5B%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%5D%2C%22timeline%22%3A%5B%22no.niva.nkm%3Aaquamonitor_hardbunn_stations%22%2C%22no.niva.nkm%3Aaquamonitor_blotbunn_stations%22%2C%22no.niva.nkm%3Anisjedata-substrat-xgbclassifier_norge%22%5D%2C%22initialCamera%22%3A%7B%22west%22%3A12.643203735351564%2C%22south%22%3A66.61130843570749%2C%22east%22%3A13.923797607421877%2C%22north%22%3A66.88104124658943%7D%2C%22homeCamera%22%3A%7B%22west%22%3A4%2C%22south%22%3A57.00000000000001%2C%22east%22%3A32%2C%22north%22%3A72%7D%2C%22viewerMode%22%3A%222d%22%2C%22showSplitter%22%3Afalse%2C%22splitPosition%22%3A0.5%2C%22settings%22%3A%7B%22baseMaximumScreenSpaceError%22%3A2%2C%22useNativeResolution%22%3Afalse%2C%22alwaysShowTimeline%22%3Afalse%2C%22baseMapId%22%3A%22basemap-openstreetmap%22%2C%22terrainSplitDirection%22%3A0%2C%22depthTestAgainstTerrainEnabled%22%3Afalse%7D%2C%22stories%22%3A%5B%5D%7D%5D%7D)

In [19]:
import geoutils as gu
import geopandas as gpd
import pandas as pd

In [20]:
# LM-DK groupings used by the substrate model (NGU Kornstorrelse ↔ NiN LM-DK).
LM_DK_TO_CLASS = {
    "DK_AB, DK_C, DK_D": "soft",
    "DK_0": "mixture",
    "DK_EFGY": "hard",
}

def summarize_points_in_bunn_types(points_gdf, polygons_gdf, column="LM_DK", mapping=None, class_order=("soft", "mixture", "hard")):
    """Per-class counts and percentages of ``points_gdf`` inside ``polygons_gdf``.

    The polygon column ``column`` (default ``LM_DK``) is aggregated through
    ``mapping`` (default: LM-DK → soft/mixture/hard) so results are reported
    consistently in the three-class scheme regardless of which raw label
    the prediction file carries. Every class in ``class_order`` is always
    present in the result, with 0 counts if that class does not occur in
    ``polygons_gdf``.
    """
    mapping = mapping or LM_DK_TO_CLASS
    polygons_gdf = polygons_gdf.copy()
    polygons_gdf["_class"] = polygons_gdf[column].map(mapping)

    total_points = len(points_gdf)
    any_poly = gpd.sjoin(points_gdf, polygons_gdf[["_class", "geometry"]], how="inner", predicate="within")
    n_in_any = len(any_poly)
    n_outside = total_points - n_in_any

    rows = []
    for cls in class_order:
        sel = polygons_gdf[polygons_gdf["_class"] == cls][["_class", "geometry"]]
        n_in = len(gpd.sjoin(points_gdf, sel, how="inner", predicate="within")) if len(sel) else 0
        rows.append({
            "class": cls,
            "n_in_type": n_in,
            "pct": n_in / total_points * 100 if total_points else 0,
            "pct_adj_for_outside": n_in / n_in_any * 100 if n_in_any else 0,
        })

    df = pd.DataFrame(rows)
    df["total_points"] = total_points
    df["n_in_any_polygon"] = n_in_any
    df["n_outside_all_polygons"] = n_outside
    return df

In [21]:
gdf_predict = gpd.read_parquet(
    "gs://niva-geodata/MarintNaturKart/results/nisjedata-substrat-xgbclassifier_norge_latest_25833.geo.parquet"
)

In [22]:
typename = "no.niva.nkm:aquamonitor_blotbunn_stations"
wfs_url_template = (
    "https://terriamap.p.niva.no/geoserver/no.niva.nkm/ows"
    "?service=WFS&version=1.0.0&request=GetFeature"
    "&typeName={}"
    "&outputFormat=application/json"
)

In [23]:
gdf_blotbunn = (
    gpd.read_file(wfs_url_template.format(typename))
    .to_crs(gdf_predict.crs)
    .drop_duplicates(subset=["geometry"])
    .reset_index(drop=True)
)

In [28]:
summarize_points_in_bunn_types(gdf_blotbunn, gdf_predict)

,class,n_in_type,pct,pct_adj_for_outside,total_points,n_in_any_polygon,n_outside_all_polygons
0,soft,930,63.394683,66.762383,1467,1393,74
1,mixture,228,15.541922,16.367552,1467,1393,74
2,hard,235,16.019087,16.870065,1467,1393,74


In [ ]:

BUNNTYPE_TO_2CLASS = {"løsbunn": "soft", "blanding": "soft", "fastbunn": "hard"}
summarize_points_in_bunn_types(gdf_blotbunn, gdf_predict, column="BunnType", mapping=BUNNTYPE_TO_2CLASS, class_order=("soft", "hard"))

,class,n_in_type,pct,pct_adj_for_outside,total_points,n_in_any_polygon,n_outside_all_polygons
0,soft,1125,76.687117,80.760948,1467,1393,74
1,hard,268,18.268575,19.239052,1467,1393,74


In [25]:
typename = "no.niva.nkm:aquamonitor_hardbunn_stations"
gdf_hardbunn = (
    gpd.read_file(wfs_url_template.format(typename))
    .to_crs(gdf_predict.crs)
    .drop_duplicates(subset=["geometry"])
    .reset_index(drop=True)
)

In [27]:
summarize_points_in_bunn_types(gdf_hardbunn, gdf_predict)

,class,n_in_type,pct,pct_adj_for_outside,total_points,n_in_any_polygon,n_outside_all_polygons
0,soft,18,6.428571,9.729730,280,185,95
1,mixture,24,8.571429,12.972973,280,185,95
2,hard,143,51.071429,77.297297,280,185,95


In [ ]:


# 2-class view for comparison with earlier reports.
summarize_points_in_bunn_types(gdf_hardbunn, gdf_predict, column="BunnType", mapping=BUNNTYPE_TO_2CLASS, class_order=("soft", "hard"))

,class,n_in_type,pct,pct_adj_for_outside,total_points,n_in_any_polygon,n_outside_all_polygons
0,soft,33,11.785714,17.837838,280,185,95
1,hard,152,54.285714,82.162162,280,185,95
